In [1]:
%matplotlib inline
%config InlineBackend.figure_format = "retina"
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# from cocoa_fisher import show_matrix, invert_matrix
# from cocoa_fisher import FisherMeta, FisherCase
# from cocoa_fisher import explore_all_variations

In [2]:
# Following mesa_apokasc/common.py, customized.
import matplotlib as mpl

def format_axis(ax: mpl.axes._axes.Axes) -> None:
    ax.minorticks_on()
    ax.xaxis.set_ticks_position('both')
    ax.yaxis.set_ticks_position('both')
    ax.patch.set_alpha(0.0)

In [3]:
real = pd.read_csv(f"cocoa_fisher_real.csv", index_col=0)
fourier = pd.read_csv(f"cocoa_fisher_fourier.csv", index_col=0)

In [4]:
# Figure 6: Weak Lensing Probes
from matplotlib.ticker import LogLocator

probes_list = [("ss", "gs", "gg"), ("gs", "gg"), ("ss",),
               ("ss", "gs"), ("gs",), ("ss", "gg"), ("gg",)]
probe_name = {"ss": "Shear", "gs": "GGL", "gg": "Clus"}
def probes2title(probes):
    if len(probes) == 3:
        return r"$3\!\times\!2$pt"
    elif len(probes) == 2:
        return probe_name[probes[0]] + " + " + probe_name[probes[1]]
    elif len(probes) == 1:
        return probe_name[probes[0]] + " Only"

label_dict = {
    "": r"W: infinitely $\bf{w}$ide priors on all (i.e., both cosmological and nuisance) parameters",
    "P": r"B: $\bf{b}$enchmark priors on photo-z bias ($\Delta_{z}^*$; $0.002$ or $0.003$) and shear bias ($m_*$; $0.005$)",
    "N": r"C: infinitely narrow priors on other studied $\bf{c}$osmological parameters ($n_{\rm s}$, $\Omega_{\rm b}$, and $h_0$)",
    "NP": r"BC: benchmark priors on $\Delta_{z}^*$ and $m_*$ (B) + infinitely narrow priors on $n_{\rm s}$, $\Omega_{\rm b}$, and $h_0$ (C)",
    "U": r"N: infinitely $\bf{n}$arrow priors on all parameters other than $\sigma_8$, $\Omega_{\rm m}$, and galaxy bias ($b^*$)"
}

In [5]:
from matplotlib.patches import ConnectionPatch

if True:  # New version, with only the 3x2pt, 2x2pt, and shear-only cases.
    fig, axs = plt.subplots(2, 7, figsize=(10.8, 3.2), sharex="all", sharey="row")

    for i_fom in range(1, 3):
        for i_row, row in enumerate([fourier.loc["base"], real.loc["base"]]):
            for i_pro, probes in enumerate(probes_list[:3]):
                # ax = axs[i_row*2+i_fom-1, i_pro]
                ax = axs[i_fom-1, i_row*4+i_pro]
                prefix = "".join(probes) + "NG"
                for i_suf, suffix in enumerate(["", "P", "N", "NP", "U"]):
                    ax.bar(i_suf, row[f"{prefix}_FoM{i_fom}{suffix}"], 1,
                           color=f"C{(i_suf-1)*2 if i_suf>2 else i_suf}",
                           hatch=[None, "/", "\\", "x", "."][i_suf], label=label_dict[suffix])

                if i_fom == 1:
                    ax.set_title(probes2title(probes))
                if i_fom == 2:
                    ax.set_xticks(np.arange(5))
                    ax.set_xticklabels(["W", "B", "C", "BC", "N"])
                ax.tick_params(right=True, which="both")

        for i_col in [0, -1]:
            ax = axs[i_fom-1, i_col]
            ax.set_ylabel([r"SNR$^2$", r"$1/\Delta^2(\sigma_8)$",
                           r"$\det^{-1/2}({\rm Cov}$"+"\n"+r"$(\sigma_8, \Omega_{\rm m}))$"][i_fom])
            if i_col == -1:
                ax.yaxis.set_label_position("right")
                continue
            ax.set_yscale("log")
            ax.yaxis.set_major_locator(LogLocator(base=10, numticks=9))
            ax.yaxis.set_minor_locator(LogLocator(base=10, subs=np.arange(2, 10)*0.1, numticks=90))

    for i in range(2):
        ax = axs[i, 3]
        if i == 1:
            for i_suf, suffix in enumerate(["", "P", "N", "NP", "U"]):
                ax.bar(i_suf, 0, 1, color=f"C{(i_suf-1)*2 if i_suf>2 else i_suf}",
                       hatch=[None, "/", "\\", "x", "."][i_suf], label=label_dict[suffix], )
            ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.2))
        ax.set_axis_off()

    axs[0, 3].text(0.25, 0.2, "Fourier\nspace", size=12,
                   ha="left", va="center", transform=axs[0, 3].transAxes)
    axs[1, 3].text(0.75, 0.8, "Real\nspace", size=12,
                   ha="right", va="center", transform=axs[1, 3].transAxes)

    for i_fom in range(1, 3):
        axs[0, 3].add_artist(ConnectionPatch(
            xyA=(1.0, 0.5), coordsA=axs[i_fom-1, 2].transAxes,
            xyB=(0.2, 0.2), coordsB=axs[0, 3].transAxes))
        axs[1, 3].add_artist(ConnectionPatch(
            xyA=(0.0, 0.5), coordsA=axs[i_fom-1, 4].transAxes,
            xyB=(0.8, 0.8), coordsB=axs[1, 3].transAxes))

    fig.subplots_adjust(hspace=0.1, wspace=0.1)
    # fig.tight_layout()
    # plt.show()
    fig.savefig('fisher_plots/probe.pdf', bbox_inches='tight')
    plt.close(fig)

In [6]:
def format_value(value):
    if value > 1e3: return f"${value:.0f}$"
    elif value > 1e2: return f"${value:.1f}$"
    elif value > 1e1: return f"${value:.2f}$"
    elif value > 1e0: return f"${value:.3f}$"
    else: return f"${value:.4f}$"

if False:  # 10/3/2025, 5/5/2026
    for i_fom in range(1, 3):
        for i_row, row in enumerate([fourier.loc["base"], real.loc["base"]]):
            print(" "*8 + "\multirow{5}{5em}{" +
                  [r"SNR$^2$", r"$1/\Delta^2(\sigma_8)$",
                   r"$\det^{-1/2}({\rm Cov}$ $(\sigma_8, \Omega_{\rm m}))$"][i_fom] +
                   r"$\times 10^{-3}$" + " (" + ["Fourier", "Real"][i_row] + ")}")

            for i_suf, suffix in enumerate(["", "P", "N", "NP", "U"]):
                # print(" "*8 + f"{i_fom}", end=" & ")
                # print("real" if i_row == 0 else "Fourier", end=" & ")
                print(" "*8 + "& "+ ["W", "B", "C", "BC", "N"][i_suf], end=" & ")

                for probes in probes_list:
                    prefix = "".join(probes) + "NG"
                    value = row[f"{prefix}_FoM{i_fom}{suffix}"]
                    # print(f"${value:.1e}$".replace("e+0", r"{\rm e}"),
                    print(format_value(value * 1e-3),
                        end=" & " if probes != probes_list[-1] else " \\\\\n")
            print(" "*4 + r"\hline")

In [7]:
# 2025.06.30 cocoa_fisher.ipynb
PROBES_LIST = [("ss", "gs", "gg"), ("gs", "gg"), ("ss",)]

def get_elem(df, probes, suffix, name="base", NG=True):
    qty = "".join(probes) + ("NG" if NG else "G") + "_" + suffix
    return df.loc[name][qty]

def get_array(df, probes, suffix, indices, name):
    arr = np.zeros(len(indices))
    for i, idx in enumerate(indices):
        arr[i] = get_elem(df, probes, suffix, name(idx))
    return arr

# 2025.08.15 paper4_plots.ipynb
def add_legend(ax, start, end):
    h, l = ax.get_legend_handles_labels()
    ax.legend(h[start:end+1], l[start:end+1], frameon=False, framealpha=0.5)  # fancybox=True

In [8]:
# Figure 7: Tomographic Bins
fig, axs = plt.subplots(4, len(PROBES_LIST), figsize=(10.8, 6.0),
                        sharex="all", sharey="row")

for i_df, df in enumerate([fourier, real]):
    for i_pro, probes in enumerate(PROBES_LIST):
        for i_row, suffix in [(1, "FoM1P"), (1, "FoM1NP"),
                              (2, "FoM2P"), (2, "FoM2NP")]:
            ax = axs[i_df*2+i_row-1, i_pro]
            ls = "-" * (1 + ("NP" in suffix))
            base = get_elem(df, probes, suffix)

            ax.axhline(1, ls=ls, c="C0", label="Benchmark")
            ax.plot(range(1, 9), get_array(df, probes, suffix, range(8), \
                lambda idx: f"tomo_no{idx}") / base, marker="o", ls=ls, c="C1", label="Exclude-one")
            ax.plot(range(1, 9), get_array(df, probes, suffix, range(8), \
                lambda idx: f"tomo_0to{idx}".replace("0to6", "no7")\
                    .replace("tomo_0to7", "base")) / base, marker=">", ls=ls, c="C2",
                    label="Cumulative\n(low to high)")
            ax.plot(range(1, 9), get_array(df, probes, suffix, range(8), \
                lambda idx: f"tomo_{idx}to7".replace("1to7", "no0")\
                    .replace("tomo_0to7", "base")) / base, marker="<", ls=ls, c="C4",
                    label="Cumulative\n(high to low)")

            if i_df == 0 and i_row == 1:
                ax.set_title(probes2title(probes))
            if i_df == 1 and i_row == 2:
                ax.set_xticks(np.arange(1, 9))
                ax.set_xlabel("Tomo bin index")
            format_axis(ax)
            ax.xaxis.set_tick_params(which="minor", bottom=False, top=False)

for i in range(4):
    ax = axs[i, 0]
    ax.set_ylabel([r"$1/\Delta^2(\sigma_8)$",
                   r"$\det^{-1/2}({\rm Cov}(\sigma_8, \Omega_{\rm m}))$"][i%2]
                  + "\nratio")
    ax.text(1.0, 0.5, ("Real space" if i >= 2 else "Fourier space") +\
        "\n(this row)", ha="left", va="center")

    ax = axs[i, -1]
    ax.plot([], [], c="k", ls="-", label='Choice "B"')
    ax.plot([], [], c="k", ls="--", label='Choice "BC"')
    match i:
        case 0: add_legend(ax, 8, 9)
        case 1: add_legend(ax, 0, 1)
        case 2: add_legend(ax, 2, 2)
        case 3: add_legend(ax, 3, 3)

# add_labels(axs, ["all bins", "jackknife", "low to high", "high to low"])
fig.tight_layout()
# plt.show()
fig.savefig('fisher_plots/tomo.pdf', bbox_inches='tight')
plt.close(fig)

In [9]:
# Figure 8: Angular Scales
fig, axs = plt.subplots(4, len(PROBES_LIST), figsize=(10.8, 6.0),
                        sharex="all", sharey="row")

for i_df, df in enumerate([fourier, real]):
    for i_pro, probes in enumerate(PROBES_LIST):
        for i_row, suffix in [(1, "FoM1P"), (1, "FoM1NP"),
                              (2, "FoM2P"), (2, "FoM2NP")]:
            ax = axs[i_df*2+i_row-1, i_pro]
            ls = "-" * (1 + ("NP" in suffix))
            base = get_elem(df, probes, suffix)

            ax.axhline(1, ls=ls, c="C0", label="Benchmark")
            ax.plot(range(1, 16), get_array(df, probes, suffix, range(15), \
                lambda idx: f"coord_no{idx}") / base, marker="o", ls=ls, c="C1", label="Exclude-one")
            ax.plot(range(3, 16), get_array(df, probes, suffix, range(2, 15), \
                lambda idx: f"coord_0to{idx}".replace("coord_0to14", "base")) / base,
                    marker=">", ls=ls, c="C2", label="Cumulative\n(low to high)")
            ax.plot(range(1, 15), get_array(df, probes, suffix, range(14), \
                lambda idx: f"coord_{idx}to14".replace("coord_0to14", "base")) / base,
                    marker="<", ls=ls, c="C4", label="Cumulative\n(high to low)")

            if i_df == 0 and i_row == 1:
                ax.set_title(probes2title(probes))
            if i_df == 1 and i_row == 2:
                ax.set_xticks(np.arange(1, 16))
                ax.set_xlabel("Scale bin index")
            format_axis(ax)
            ax.xaxis.set_tick_params(which="minor", bottom=False, top=False)

for i in range(4):
    ax = axs[i, 0]
    ax.set_ylabel([r"$1/\Delta^2(\sigma_8)$",
                   r"$\det^{-1/2}({\rm Cov}(\sigma_8, \Omega_{\rm m}))$"][i%2]
                  + "\nratio")
    if i >= 2:
        ax.text(15.0, 0.5, "Real space\n(this row)", ha="right", va="center")
    else:
        ax.text(1.0, 0.5, "Fourier space\n(this row)", ha="left", va="center")

    ax = axs[i, 1]
    ax.plot([], [], c="k", ls="-", label='Choice "B"')
    ax.plot([], [], c="k", ls="--", label='Choice "BC"')
    match i:
        case 0: add_legend(ax, 8, 9)
        case 1: add_legend(ax, 0, 1)
        case 2: add_legend(ax, 2, 2)
        case 3: add_legend(ax, 3, 3)

# add_labels(axs, ["all bins", "jackknife", "low to high", "high to low"])
fig.tight_layout()
# plt.show()
fig.savefig('fisher_plots/scale.pdf', bbox_inches='tight')
plt.close(fig)

In [10]:
# Figure 9: Super-Sample Covariance
fig, axs = plt.subplots(2, len(PROBES_LIST), figsize=(10.8, 3.6),
                        sharex="all", sharey="row")
coefs = np.linspace(0.0, 1.0, 11)

for i_df, df in enumerate([fourier, real]):
    for i_pro, probes in enumerate(PROBES_LIST):
        for i_row, suffix in [(1, "FoM1P"), (1, "FoM1NP"),
                              (2, "FoM2P"), (2, "FoM2NP")]:
            ax = axs[i_df, i_pro]
            ls = "-" * (1 + ("NP" in suffix))
            base = get_elem(df, probes, suffix)

            arr = np.zeros_like(coefs)
            arr[-1] = get_elem(df, probes, suffix)
            ax.axhline(1, ls=ls, c="C0")

            arr[1:-1] = get_array(df, probes, suffix, coefs[1:-1], \
                lambda idx: f"coef_ng_{idx:.1f}")
            arr[0] = get_elem(df, probes, suffix, NG=False)
            ax.plot(coefs, arr / base, marker=" oD"[i_row], ls=ls, c=f"C{i_row}")

            if i_df == 0 and i_row == 1:
                ax.set_title(probes2title(probes))
            if i_df == 1 and i_row == 2:
                ax.set_xlabel(r"Cov$^{\rm NG}$ coefficient")
            format_axis(ax)

for i in range(2):
    ax = axs[i, 0]
    ax.set_ylabel("Ratio")
    ax.text(0.95, 0.95, ["Fourier", "Real"][i] + " space\n(this row)",
            ha="right", va="top", transform=ax.transAxes)

    ax = axs[i, 1]
    ax.plot([], [], c="C1", marker="o", ls="-", label=r"$1/\Delta^2(\sigma_8)$")
    ax.plot([], [], c="C2", marker="D", ls="-", label=r"$\det^{-1/2}({\rm Cov}(\sigma_8, \Omega_{\rm m}))$")
    ax.plot([], [], c="k", ls="-", label='Choice "B"')
    ax.plot([], [], c="k", ls="--", label='Choice "BC"')
    match i:
        case 0: add_legend(ax, 2, 3)
        case 1: add_legend(ax, 0, 1)

fig.tight_layout()
# plt.show()
fig.savefig('fisher_plots/ssc.pdf', bbox_inches='tight')
plt.close(fig)

In [11]:
# Figures 10 and 12: Scaling of Priors
scales = np.geomspace(0.1, 10, 9)

def explore_nuisance_scaling(real, fourier, prefix, figname=None):
    fig, axs = plt.subplots(4, len(PROBES_LIST), figsize=(10.8, 6.0),
                            sharex="all", sharey="row")

    for i_df, df in enumerate([fourier, real]):
        for i_pro, probes in enumerate(PROBES_LIST):
            for i_row, suffix in [(1, "FoM1P"), (1, "FoM1NP"),
                                  (2, "FoM2P"), (2, "FoM2NP")]:
                ax = axs[i_df*2+i_row-1, i_pro]
                ls = "-" * (1 + ("NP" in suffix))
                arr = get_array(df, probes, suffix, scales, \
                    lambda idx: f"{prefix[6:]}_var_all_{idx:.1f}")
                base = arr[4]

                ax.axhline(1, ls=ls, c="C0", label="Benchmark")
                ax.plot(scales, arr / base, marker="o", ls=ls, c="C1", label="All tomo bins")
                # if len(probes) == 3:
                #     print(["Fourier", "Real"][i_df], prefix, suffix, (arr / base)[3:6])
                if prefix == "roman_DZ_S":
                    ax.plot(scales, get_array(df, probes, suffix, scales, \
                        lambda idx: f"{prefix[6:]}_var_0_{idx:.1f}") / base,
                            marker="^", ls=ls, c="C2", label="Single tomo bin\n(lowest $z$)")
                ax.plot(scales, get_array(df, probes, suffix, scales, \
                    lambda idx: f"{prefix[6:]}_var_7_{idx:.1f}") / base,
                        marker="v", ls=ls, c="C4", label="Single tomo bin\n(highest $z$)")

                if i_df == 0 and i_row == 1:
                    ax.set_title(probes2title(probes))
                if i_df == 1 and i_row == 2:
                    ax.set_xlabel(r"Scaling factor")
                    ax.set_xscale("log")
                format_axis(ax)

    for i in range(4):
        ax = axs[i, 0]
        ax.set_ylabel([r"$1/\Delta^2(\sigma_8)$",
                    r"$\det^{-1/2}({\rm Cov}(\sigma_8, \Omega_{\rm m}))$"][i%2]
                    + "\nratio")
        ax.text(0.05, 0.05, ["Fourier", "Real"][i//2] + " space\n(this row)",
                ha="left", va="bottom", transform=ax.transAxes)

        ax = axs[i, 1]
        ax.plot([], [], c="k", ls="-", label='Choice "B"')
        ax.plot([], [], c="k", ls="--", label='Choice "BC"')

        if prefix == "roman_DZ_S":
            match i:
                case 0: add_legend(ax, 8, 9)
                case 1: add_legend(ax, 0, 1)
                case 2: add_legend(ax, 2, 2)
                case 3: add_legend(ax, 3, 3)
        elif prefix == "roman_M":
            match i:
                case 0: add_legend(ax, 6, 7)
                case 1: add_legend(ax, 0, 1)
                case 2: add_legend(ax, 2, 2)

    fig.tight_layout()
    if figname is None:
        plt.show()
    else:
        fig.savefig(f'fisher_plots/{figname}.pdf', bbox_inches='tight')
        plt.close(fig)

explore_nuisance_scaling(real, fourier, "roman_DZ_S", figname="photoz_scale")
explore_nuisance_scaling(real, fourier, "roman_M", figname="shear_scale")

In [12]:
# Figures 11 and 13: Correlation of Priors
corrs = np.linspace(-0.9, 0.9, 19)
corrs_all = corrs[8:]
corrs_pair = corrs[4:-4]

def explore_nuisance_correlation(real, fourier, prefix, figname=None):
    fig, axs = plt.subplots(4, len(PROBES_LIST), figsize=(10.8, 6.0),
                            sharex="all", sharey="row")

    for i_df, df in enumerate([fourier, real]):
        for i_pro, probes in enumerate(PROBES_LIST):
            for i_row, suffix in [(1, "FoM1P"), (1, "FoM1NP"),
                                (2, "FoM2P"), (2, "FoM2NP")]:
                ax = axs[i_df*2+i_row-1, i_pro]
                ls = "-" * (1 + ("NP" in suffix))
                arr = get_array(df, probes, suffix, corrs_all, \
                    lambda idx: f"{prefix[6:]}_corr_all_{idx:.1f}"\
                        .replace(f"{prefix[6:]}_corr_all_0.0", "base"))
                base = arr[1]

                ax.axhline(1, ls=ls, c="C0", label="Benchmark")
                ax.plot(corrs_all, arr / base, marker="o", ls=ls, c="C1", label="All bin pairs")
                ax.plot(corrs_pair, get_array(df, probes, suffix, corrs_pair, \
                    lambda idx: f"{prefix[6:]}_corr_pair_{idx:.1f}"\
                        .replace(f"{prefix[6:]}_corr_pair_0.0", "base")) / base,
                        marker="D", ls=ls, c="C2", label="Adjacent bin pairs")

                if i_df == 0 and i_row == 1:
                    ax.set_title(probes2title(probes))
                if i_df == 1 and i_row == 2:
                    ax.set_xlabel(r"Correlation coefficient")
                format_axis(ax)

    for i in range(4):
        ax = axs[i, 0]
        ax.set_ylabel([r"$1/\Delta^2(\sigma_8)$",
                    r"$\det^{-1/2}({\rm Cov}(\sigma_8, \Omega_{\rm m}))$"][i%2]
                    + "\nratio")
        ax.text(0.50, 0.95, ["Fourier", "Real"][i//2] + " space\n(this row)",
                ha="center", va="top", transform=ax.transAxes)

        ax = axs[i, 1]
        ax.plot([], [], c="k", ls="-", label='Choice "B"')
        ax.plot([], [], c="k", ls="--", label='Choice "BC"')
        match i:
            case 0: add_legend(ax, 6, 7)
            case 1: add_legend(ax, 0, 1)
            case 2: add_legend(ax, 2, 2)

    fig.tight_layout()
    if figname is None:
        plt.show()
    else:
        fig.savefig(f'fisher_plots/{figname}.pdf', bbox_inches='tight')
        plt.close(fig)

explore_nuisance_correlation(real, fourier, "roman_DZ_S", figname="photoz_corr")
explore_nuisance_correlation(real, fourier, "roman_M", figname="shear_corr")